In [9]:
import pandas as pd
import re

#Raw RPK matrix: peptides x sample IDs

cohort_df = pd.read_csv("./preprocessing/earliest_tp_rpk_avg_value_for_cohort_techreps_071626.csv").set_index("peptide")

#sample metadata with time info
meta = pd.read_csv("./preprocessing/master_sample_metadata_060626.csv")

#import df_hbdb
hbdb_df = pd.read_csv("./preprocessing/hbdb_pass_100k_techreps_averaged_071626.csv").set_index("peptide")

#load peptide protein metadata
protein_metadata = pd.read_csv("lassa_library_sequences_species_standardized_070726.csv")

# making a long format RPK dataframe

In [10]:
#Long format for merging with metadata
exposed_rpk_long = (
    cohort_df.reset_index()
    .melt(id_vars="peptide", var_name="sample_id", value_name="rpk")
    .merge(
        meta[[
            "sample_id", "participant_id",
            "collection_date", "collection_year", "visit_number", 
            "years_since_baseline", "is_baseline"
        ]],
        on="sample_id",
        how="left"
    )
).set_index('peptide')

In [11]:
#attach peptide taxonomy metadata to baseline dataframe
meta_ann_baseline = (
    protein_metadata[["seq_id", "family", "genus", "species"]]
    .drop_duplicates("seq_id")
    .set_index("seq_id")
)

In [12]:
exposed_rpk_long = exposed_rpk_long.drop(columns=["family", "genus", "species"], errors="ignore").join(
    meta_ann_baseline[["family", "genus", "species"]],
    how="left"
)

In [13]:
import numpy as np
exposed_rpk_long['status'] = np.where(
    exposed_rpk_long['participant_id'].str.startswith('C'), 
    'Contact', 
    np.where(
        exposed_rpk_long['participant_id'].str.startswith('S'), 
        'Survivor', 
        None
    )
)

In [14]:
exposed_rpk_long['sample_id'].nunique()

397

# Normalize to US healthies using z-score

In [15]:
import numpy as np

mat = cohort_df.apply(pd.to_numeric, errors="coerce").fillna(0.0) #converts every column to a numeric type, if cannot be converted, fills with Nan, but then later NaN gets converted to 0.0

#log transforming before z-scoring
log_mat = np.log10(mat + 1.0)

In [16]:
log_hbdb = np.log10(hbdb_df + 1.0)

log_hbdb_mean = log_hbdb.mean(axis=1)
log_hbdb_std = log_hbdb.std(axis=1, ddof=1).replace(0, np.nan)

In [17]:
#z-score using healthy controls as reference
z_mat = log_mat.sub(log_hbdb_mean, axis=0).div(log_hbdb_std, axis=0)
z_mat = z_mat.fillna(0.0)

In [18]:
z_mat.to_csv('z_score_df_SL_over_US_071626.csv')

# Making a z_score dataframe in a long format

In [19]:
#Long format for merging with metadata

z_long = (
    z_mat.reset_index()
    .melt(id_vars="peptide", var_name="sample_id", value_name="z_score")
    .merge(
        meta[[
            "sample_id", "participant_id",
            "collection_date", "collection_year", "visit_number", 
            "years_since_baseline", "is_baseline"
        ]],
        on="sample_id",
        how="left"
    )
).set_index('peptide')

In [20]:
#adding genus to the z_long table

z_long = (
    z_long.reset_index()
    .merge(
        protein_metadata[["seq_id", "genus"]],
        left_on="peptide", 
        right_on="seq_id", 
        how="left"
    )
    .drop(columns="seq_id")
).set_index('peptide')

In [21]:
z_long.to_csv("long_df_z_score_SL_over_US.csv")

# MWU to identify statistically significant peptides

In [22]:
# Peptide-level reactivity: Lassa-exposed (Survivor + Contact) at all timepoints vs HBDB
##
## HBDB comparator uses leave-one-out (LOO) z-scores on the same log scale
## as exposed z-scores to keep both groups comparable.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# -- parameters
Z_THRESH = 3.0
SIG_FDR = 0.05
MIN_N_PER_GROUP = 3

# -- peptide universe
all_peps = (
    z_mat.index
    .intersection(z_mat.index)
    .intersection(log_hbdb.index)
)

print(f"peptides avail for downstream analysis: {len(all_peps) }")

if len(all_peps) == 0:
    raise ValueError("no overlapping peptides found across z_mat, baseline_z_mat, and log_hbdb")

# -- HBDB LOO z-scores
hbdb_cols_z = list(log_hbdb.columns)
hbdb_kept = log_hbdb.loc[all_peps, hbdb_cols_z].astype(float).values  # (n_kept, n_hbdb)
n_hbdb_s = hbdb_kept.shape[1]

z_hbdb_loo = np.full_like(hbdb_kept, np.nan, dtype=float)
for i in range(n_hbdb_s):
    idx = np.delete(np.arange(n_hbdb_s), i)
    mu = hbdb_kept[:, idx].mean(axis=1)
    sd = hbdb_kept[:, idx].std(axis=1, ddof=1)
    sd = np.where(sd == 0, np.nan, sd)
    with np.errstate(invalid='ignore', divide='ignore'):
        z_hbdb_loo[:, i] = (hbdb_kept[:, i] - mu) / sd

z_hbdb_loo_df = pd.DataFrame(z_hbdb_loo, index=all_peps, columns=hbdb_cols_z)
print(f"HBDB LOO z-scores computed: {n_hbdb_s} samples x {len(all_peps):,} peptides")

# -- MWU per peptide: all exposed vs HBDB LOO
rows = []
for pep in all_peps:
    e_vals = z_mat.loc[pep].dropna().values
    h_vals = z_hbdb_loo_df.loc[pep].dropna().values
    if len(e_vals) < MIN_N_PER_GROUP or len(h_vals) < MIN_N_PER_GROUP:
        continue
    _, p = mannwhitneyu(e_vals, h_vals, alternative='two-sided')
    rows.append({
        'peptide': pep,
        'median_z_exposed': float(np.median(e_vals)),
        'median_z_hbdb': float(np.median(h_vals)),
        'n_exposed': int(len(e_vals)),
        'n_hbdb': int(len(h_vals)),
        'pval': float(p),
    })

res_exposed_hbdb = pd.DataFrame(rows)
if res_exposed_hbdb.empty:
    raise ValueError(
        "No peptides had sufficient observations in both groups for MWU. "
        "Check MIN_N_PER_GROUP or upstream missingness."
    )
res_exposed_hbdb = res_exposed_hbdb.set_index('peptide')

# -- multiple testing and effect size 
_, res_exposed_hbdb['pval_adj'], _, _ = multipletests(res_exposed_hbdb['pval'], method='fdr_bh')
res_exposed_hbdb['delta_z'] = res_exposed_hbdb['median_z_exposed'] - res_exposed_hbdb['median_z_hbdb']
res_exposed_hbdb['neg_log10_p'] = -np.log10(res_exposed_hbdb['pval_adj'].clip(lower=1e-300))

# -- baseline fraction enriched -- first exposure point (used later for filtering and hit calling)
res_exposed_hbdb['frac_enriched'] = (
    (z_mat.loc[res_exposed_hbdb.index] > Z_THRESH).mean(axis=1)
)

peptides avail for downstream analysis: 90132
HBDB LOO z-scores computed: 87 samples x 90,132 peptides


In [23]:
# -------------------- metadata annotation --------------------
meta_ann_hbdb = (
    protein_metadata[['seq_id', 'sequence', 'species', 'genus', 'family', 'accession', 'protein_name', 'fragment']]
    .drop_duplicates('seq_id')
    .set_index('seq_id')
)

res_exposed_hbdb[['sequence', 'species', 'genus', 'family', 'accession', 'protein_name', 'fragment']] = (
    meta_ann_hbdb.reindex(res_exposed_hbdb.index)
    [['sequence', 'species', 'genus', 'family','accession', 'protein_name', 'fragment']]
)

res_exposed_hbdb['is_mm'] = res_exposed_hbdb['genus'] == 'Mammarenavirus'

# -------------------- summary --------------------
sig_exp = res_exposed_hbdb[res_exposed_hbdb['pval_adj'] < SIG_FDR]
print(
    f"\nAll Lassa-exposed longitudinal vs HBDB - tested: {len(res_exposed_hbdb):,}  "
    f"FDR < {SIG_FDR}: {len(sig_exp):,}"
)
print(f"  Exposed-enriched (delta_z > 0): {(sig_exp['delta_z'] > 0).sum()}")
print(f"  HBDB-enriched    (delta_z < 0): {(sig_exp['delta_z'] < 0).sum()}")



All Lassa-exposed longitudinal vs HBDB - tested: 88,295  FDR < 0.05: 53,300
  Exposed-enriched (delta_z > 0): 36754
  HBDB-enriched    (delta_z < 0): 16546


In [24]:
# ------ stringent filter: MWU + FDR + exposure-specific (for final plots)

sig_peps = res_exposed_hbdb[
    (res_exposed_hbdb['pval_adj'] < SIG_FDR) &
    (res_exposed_hbdb['delta_z']  > 0) #significant for exposed
].index

In [25]:
import pandas as pd

pd.Series(sig_peps).to_csv("indices_peps_sig_pos_d_z_071626.csv", index=False, header=None)

In [26]:
z_hbdb_loo_df.to_csv("z_hbdb_loo_results_071626.csv", index=False, header=None)